# خَلَف | KHALAF-MUB-001-4B

**Inference notebook · Research prototype · v0.4**

This notebook loads **existing** QLoRA adapter weights from Google Drive; it does **not** train a new model. DAAD (ضاد) is the future web app; KHALAF (خَلَف) is the language model.

**Requirements:** a Colab CUDA GPU (such as T4), access to the original adapter directory, and permission to use the base model. If GPU is unavailable, file checks still work but text generation is skipped. No quality guarantees.

## 1 · Environment and saved artifacts

Select **Runtime → Change runtime type → T4 GPU** for inference. The Drive folder name is intentionally unchanged to preserve compatibility with the saved training run.

In [ ]:
# @title 1 — Mount Drive and inspect model files
from pathlib import Path
from google.colab import drive
import json
import torch

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

RUN_NAME = "MUB_001_Qwen3_4B_v04" # @param {type:"string"}
RUN = Path("/content/drive/MyDrive/DaadUnderstand") / RUN_NAME
ADAPTER = RUN / "final_adapter"

assert (RUN / "experiment.json").is_file(), f"Missing: {RUN / 'experiment.json'}"
assert (ADAPTER / "adapter_config.json").is_file(), "Missing adapter_config.json"
assert (ADAPTER / "adapter_model.safetensors").is_file(), "Missing trained adapter weights"

meta = json.loads((RUN / "experiment.json").read_text(encoding="utf-8"))
assert meta.get("base") and meta.get("revision") and meta.get("system"), "Training metadata incomplete"

print("KHALAF-MUB-001-4B | trained adapter found")
print("Base:", meta["base"])
print("GPU available:", torch.cuda.is_available())
print("Inference:", "available after setup" if torch.cuda.is_available() else "requires GPU; skipped")

## 2 · Install dependencies (GPU only)

The notebook avoids executing generation on CPU because loading a 4B base model would be slow and memory-heavy. If a dependency installation changes the active Transformers version, restart the runtime and rerun the notebook from the top.

In [ ]:
# @title 2 — Install inference dependencies
import sys
import subprocess
if torch.cuda.is_available():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers==4.55.4", "bitsandbytes==0.50.2",
         "peft", "accelerate", "safetensors"],
        check=True
    )
    print("Dependencies installed. Restart runtime if the environment was already imported.")
else:
    print("GPU unavailable: installation and inference skipped.")

## 3 · Translate a dialect sentence

The code uses deterministic greedy generation (`do_sample=False`) rather than the model's sampling defaults. It checks the environment before importing large inference libraries, and loads the adapter only once per runtime. This is an exploratory translation, not a validated result.

In [ ]:
# @title 3 — KHALAF: dialect to MSA
TEXT = "استنانيا شوي" # @param {type:"string"}

if not torch.cuda.is_available():
    print("KHALAF-MUB-001-4B")
    print("Model weights: Found")
    print("GPU: Unavailable")
    print("Inference test: Skipped (no training performed)")
else:
    import gc
    from copy import deepcopy
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    key = str(ADAPTER.resolve())
    if globals().get("_khalaf_loaded_adapter") != key:
        globals().pop("khalaf_model", None)
        gc.collect()
        torch.cuda.empty_cache()

        khalaf_tokenizer = AutoTokenizer.from_pretrained(
            str(ADAPTER), trust_remote_code=False
        )
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        base = AutoModelForCausalLM.from_pretrained(
            meta["base"],
            revision=meta["revision"],
            quantization_config=quant,
            device_map={"": 0},
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
            trust_remote_code=False
        )
        khalaf_model = PeftModel.from_pretrained(base, str(ADAPTER)).eval()
        _khalaf_loaded_adapter = key

    messages = [
        {"role": "system", "content": meta["system"]},
        {"role": "user", "content": TEXT}
    ]
    prompt = khalaf_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = khalaf_tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to("cuda")

    cfg = deepcopy(khalaf_model.generation_config)
    cfg.do_sample = False
    cfg.num_beams = 1
    cfg.temperature = None
    cfg.top_p = None
    cfg.top_k = None
    cfg.max_new_tokens = 128
    cfg.pad_token_id = khalaf_tokenizer.pad_token_id
    cfg.use_cache = True

    with torch.inference_mode():
        output = khalaf_model.generate(
            **inputs, generation_config=cfg, use_model_defaults=False
        )
    prediction = khalaf_tokenizer.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("Model: KHALAF-MUB-001-4B")
    print("العامية المبرزية:", TEXT)
    print("الفصحى:", prediction)

## Notes

- This notebook deliberately leaves the historical Drive path `DaadUnderstand/MUB_001_Qwen3_4B_v04` unchanged.
- It contains **no embedded dataset, uploaded files, access tokens, API keys, outputs, or training loops**.
- Model weights are not included in this notebook or repository.
- Linguistic-engine experiments and evaluation audits are described in the repository documents rather than run automatically.
- The only supported inference setup here is a CUDA GPU with quantization. **CPU skips generation without raising SystemExit.**